In [48]:
from ucimlrepo import fetch_ucirepo 
bank_marketing = fetch_ucirepo(id=222) 


In [49]:
# data (as pandas dataframes) 
x = bank_marketing.data.features 
y = bank_marketing.data.targets 

# metadata 
print(bank_marketing.metadata) 

# variable information 
print(bank_marketing.variables) 

{'uci_id': 222, 'name': 'Bank Marketing', 'repository_url': 'https://archive.ics.uci.edu/dataset/222/bank+marketing', 'data_url': 'https://archive.ics.uci.edu/static/public/222/data.csv', 'abstract': 'The data is related with direct marketing campaigns (phone calls) of a Portuguese banking institution. The classification goal is to predict if the client will subscribe a term deposit (variable y).', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 45211, 'num_features': 16, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Occupation', 'Marital Status', 'Education Level'], 'target_col': ['y'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2014, 'last_updated': 'Fri Aug 18 2023', 'dataset_doi': '10.24432/C5K306', 'creators': ['S. Moro', 'P. Rita', 'P. Cortez'], 'intro_paper': {'ID': 277, 'type': 'NATIVE', 'title': 'A data-driven approach to predict the s

In [50]:
x.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   age          45211 non-null  int64
 1   job          44923 non-null  str  
 2   marital      45211 non-null  str  
 3   education    43354 non-null  str  
 4   default      45211 non-null  str  
 5   balance      45211 non-null  int64
 6   housing      45211 non-null  str  
 7   loan         45211 non-null  str  
 8   contact      32191 non-null  str  
 9   day_of_week  45211 non-null  int64
 10  month        45211 non-null  str  
 11  duration     45211 non-null  int64
 12  campaign     45211 non-null  int64
 13  pdays        45211 non-null  int64
 14  previous     45211 non-null  int64
 15  poutcome     8252 non-null   str  
dtypes: int64(7), str(9)
memory usage: 7.3 MB


In [51]:
from sklearn.model_selection import train_test_split,KFold,cross_val_score
from sklearn.preprocessing import LabelEncoder,MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

In [52]:
x

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,no,825,no,no,cellular,17,nov,977,3,-1,0,NaN
45207,71,retired,divorced,primary,no,1729,no,no,cellular,17,nov,456,2,-1,0,NaN
45208,72,retired,married,secondary,no,5715,no,no,cellular,17,nov,1127,5,184,3,success
45209,57,blue-collar,married,secondary,no,668,no,no,telephone,17,nov,508,4,-1,0,NaN


In [53]:
y

,y
0,no
1,no
2,no
3,no
4,no
...,...
45206,yes
45207,yes
45208,yes
45209,no


In [54]:
x.isnull().sum()

age                0
job              288
marital            0
education       1857
default            0
balance            0
housing            0
loan               0
contact        13020
day_of_week        0
month              0
duration           0
campaign           0
pdays              0
previous           0
poutcome       36959
dtype: int64

In [55]:
y.isnull().sum()

y    0
dtype: int64

In [56]:
x.shape

(45211, 16)

In [57]:
y.shape

(45211, 1)

In [58]:
x.drop_duplicates()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,no,825,no,no,cellular,17,nov,977,3,-1,0,NaN
45207,71,retired,divorced,primary,no,1729,no,no,cellular,17,nov,456,2,-1,0,NaN
45208,72,retired,married,secondary,no,5715,no,no,cellular,17,nov,1127,5,184,3,success
45209,57,blue-collar,married,secondary,no,668,no,no,telephone,17,nov,508,4,-1,0,NaN


In [59]:
import warnings
warnings.filterwarnings('ignore')

In [60]:
x['job']= x['job'].fillna(x['job'].mode()[0],inplace=True)

In [61]:
x['education'] = x['education'].fillna(x['education'].mode()[0],inplace=True)

In [62]:
x

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN
3,47,blue-collar,married,secondary,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN
4,33,blue-collar,single,secondary,no,1,no,no,NaN,5,may,198,1,-1,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,no,825,no,no,cellular,17,nov,977,3,-1,0,NaN
45207,71,retired,divorced,primary,no,1729,no,no,cellular,17,nov,456,2,-1,0,NaN
45208,72,retired,married,secondary,no,5715,no,no,cellular,17,nov,1127,5,184,3,success
45209,57,blue-collar,married,secondary,no,668,no,no,telephone,17,nov,508,4,-1,0,NaN


In [63]:
x.isnull().sum()

age                0
job                0
marital            0
education          0
default            0
balance            0
housing            0
loan               0
contact        13020
day_of_week        0
month              0
duration           0
campaign           0
pdays              0
previous           0
poutcome       36959
dtype: int64

In [64]:
x.drop(columns=['poutcome','contact'],inplace=True)

In [65]:
x.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'day_of_week', 'month', 'duration', 'campaign', 'pdays',
       'previous'],
      dtype='str')

In [66]:
scaler = MinMaxScaler()
x[['age','balance','duration']] = scaler.fit_transform(x[['age','balance','duration']])

In [67]:
x

,age,job,marital,education,default,balance,housing,loan,day_of_week,month,duration,campaign,pdays,previous
0,0.519481,management,married,tertiary,no,0.092259,yes,no,5,may,0.053070,1,-1,0
1,0.337662,technician,single,secondary,no,0.073067,yes,no,5,may,0.030704,1,-1,0
2,0.194805,entrepreneur,married,secondary,no,0.072822,yes,yes,5,may,0.015453,1,-1,0
3,0.376623,blue-collar,married,secondary,no,0.086476,yes,no,5,may,0.018707,1,-1,0
4,0.194805,blue-collar,single,secondary,no,0.072812,no,no,5,may,0.040260,1,-1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,0.428571,technician,married,tertiary,no,0.080293,no,no,17,nov,0.198658,3,-1,0
45207,0.688312,retired,divorced,primary,no,0.088501,no,no,17,nov,0.092721,2,-1,0
45208,0.701299,retired,married,secondary,no,0.124689,no,no,17,nov,0.229158,5,184,3
45209,0.506494,blue-collar,married,secondary,no,0.078868,no,no,17,nov,0.103294,4,-1,0


In [68]:
lab_enc = LabelEncoder()

In [69]:
import joblib
col =['housing','loan','month','default','education','marital','job']
encoder ={}
for i in col:
    x[i] = lab_enc.fit_transform(x[i])
    encoder[i] = lab_enc
x

,age,job,marital,education,default,balance,housing,loan,day_of_week,month,duration,campaign,pdays,previous
0,0.519481,4,1,2,0,0.092259,1,0,5,8,0.053070,1,-1,0
1,0.337662,9,2,1,0,0.073067,1,0,5,8,0.030704,1,-1,0
2,0.194805,2,1,1,0,0.072822,1,1,5,8,0.015453,1,-1,0
3,0.376623,1,1,1,0,0.086476,1,0,5,8,0.018707,1,-1,0
4,0.194805,1,2,1,0,0.072812,0,0,5,8,0.040260,1,-1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,0.428571,9,1,2,0,0.080293,0,0,17,9,0.198658,3,-1,0
45207,0.688312,5,0,0,0,0.088501,0,0,17,9,0.092721,2,-1,0
45208,0.701299,5,1,1,0,0.124689,0,0,17,9,0.229158,5,184,3
45209,0.506494,1,1,1,0,0.078868,0,0,17,9,0.103294,4,-1,0


In [70]:
print(encoder)

{'housing': LabelEncoder(), 'loan': LabelEncoder(), 'month': LabelEncoder(), 'default': LabelEncoder(), 'education': LabelEncoder(), 'marital': LabelEncoder(), 'job': LabelEncoder()}


In [71]:
joblib.dump(encoder,open("encoder.pkl", "wb"))

In [72]:
y_enc = LabelEncoder()
y= y_enc.fit_transform(y['y'])
y

array([0, 0, 0, ..., 1, 0, 0], shape=(45211,))

In [73]:
joblib.dump(y_enc,open("y.pkl", "wb"))

In [74]:
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.3)

In [75]:
X_train.shape,X_test.shape,y_train.shape,y_test.shape

((31647, 14), (13564, 14), (31647,), (13564,))

In [76]:
dtc_model = DecisionTreeClassifier()

In [77]:
kfold = KFold(n_splits=5)
vali_score = cross_val_score(
    dtc_model,X_train,y_train,cv=kfold
)

In [78]:
print(vali_score.mean()*100)
print(vali_score)

86.6812258923978
[0.86682464 0.85924171 0.8668036  0.87565176 0.86553958]


In [79]:
dtc_model.fit(X_train,y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curre

In [80]:
y_pred = dtc_model.predict(X_test)
y_pred[:10]

array([0, 0, 0, 1, 0, 1, 0, 0, 0, 0])

In [81]:
y_test[:10]

array([0, 0, 0, 0, 0, 1, 0, 0, 0, 1])

In [82]:
from sklearn.metrics import accuracy_score,confusion_matrix
print(accuracy_score(y_test,y_pred))

0.8659687407844294


In [83]:
print(confusion_matrix(y_test,y_pred))

[[11044   920]
 [  898   702]]


In [84]:
joblib.dump(dtc_model,'Dtc_model.pkl')

['Dtc_model.pkl']

In [85]:
x.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'day_of_week', 'month', 'duration', 'campaign', 'pdays',
       'previous'],
      dtype='str')

In [86]:
x

,age,job,marital,education,default,balance,housing,loan,day_of_week,month,duration,campaign,pdays,previous
0,0.519481,4,1,2,0,0.092259,1,0,5,8,0.053070,1,-1,0
1,0.337662,9,2,1,0,0.073067,1,0,5,8,0.030704,1,-1,0
2,0.194805,2,1,1,0,0.072822,1,1,5,8,0.015453,1,-1,0
3,0.376623,1,1,1,0,0.086476,1,0,5,8,0.018707,1,-1,0
4,0.194805,1,2,1,0,0.072812,0,0,5,8,0.040260,1,-1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,0.428571,9,1,2,0,0.080293,0,0,17,9,0.198658,3,-1,0
45207,0.688312,5,0,0,0,0.088501,0,0,17,9,0.092721,2,-1,0
45208,0.701299,5,1,1,0,0.124689,0,0,17,9,0.229158,5,184,3
45209,0.506494,1,1,1,0,0.078868,0,0,17,9,0.103294,4,-1,0
